# Supervised ML PipeLine Template

This notebook contains a comprehensive ML pipeline template for the following model:
2) XGBoost

Additonally, the follow undersample methods can be chosen:
1) RandomUnderSampler

### Some of the assumptions: 
1) Data is pre-cleaned with no missing values.
2) Only numeric features are used as all other columns are excluded automatically.

## How will this be used (steps):
1) Only run in new notebooks, import the module as follows: **from Supervised_ML_Pipeline import MLPipeline**
2) Declare an object and initialize it as follows: **pipe = MLPipeline(model_name="svm", target_col="outcome")**
3) This line does the data spliting, training & testing and provides results: **pipe.run(df, param_grid={}, group_col="", exclude_cols=[], undersample_method=None)**

Important - Read the below parameters

#### Required parameters:
1) model_name - can't run without knowing which model
2) target_col - can't extract features without knowing the target
3) param_grid - can't run GridSearchCV without parameters
4) group_col - can't do StratifiedGroupKFold without groups

#### Optional parameters:
1) exclude_cols - defaults to [] if not passed, pipeline still runs fine
2) undersample_method - defaults to None, no undersampling applied
3) output_dir - defaults to ".", saves in current directory
4) cv, scoring, test_size, random_state, verbose - all have sensible defaults (5, average_precision, 0.4, 42, 1)

#### Additional stuff:
1) list_models() gives a list of models available
2) list_undersample_methods() gives a list of undersampling methods available

## Results:
1) Displays a quick summary of the training along with evaluation metrics conducted on test data.
2) Saves a score distrution plot and confusion matrix plot as png.
3) Saves a csv file with test scores and their respective risk label.
4) Saves a summary, waterfall and feature importance plot as png (SHAP).


In [1]:
# In the Libraries portion of a new different notebook
from Supervised_ML_Pipeline import MLPipeline, add_inne_features
import pandas as pd

In [2]:
MLPipeline.list_models()

['logistic_regression', 'xgboost', 'svm']

In [3]:
MLPipeline.list_undersample_methods()

['RandomUnderSampler', 'CLEANSE']

In [4]:
MLPipeline.list_feature_selection_methods()

['k_best_catboost', 'k_best_xgb', 'k_best_lgbm', 'k_best_decision_tree']

In [5]:
select_methods = MLPipeline.list_feature_selection_methods()
select_methods

['k_best_catboost', 'k_best_xgb', 'k_best_lgbm', 'k_best_decision_tree']

In [6]:
df = pd.read_csv("/dsa/groups/casestudycf25/team02/silver/unified_dataset.csv")

target_col = 'target'

exclude_cols = [
    'npi', 'rfrg_prvdr_state_abrvtn',
    'specialty_type', 'specialty'
]

df.head(5)

,npi,rfrg_prvdr_state_abrvtn,year,target,avg_suplr_mdcr_pymt_amt_sum,tot_suplr_mdcr_pymt_amt_hcpcs_rentl_ind_z_scr_sum,tot_suplr_mdcr_pymt_amt_hcpcs_rentl_ind_z_scr_median,avg_suplr_mdcr_alowd_amt_hcpcs_rentl_ind_rat_median,avg_suplr_mdcr_pymt_amt_min,tot_suplr_mdcr_pymt_amt_hcpcs_rentl_ind_z_scr_min,...,claims_per_bene_zscore_by_type,claims_per_bene_outlier_by_type,services_per_bene_outlier,services_per_bene_zscore_by_type,services_per_bene_outlier_by_type,benes_per_supplier_outlier,benes_per_supplier_zscore_by_type,benes_per_supplier_outlier_by_type,elderly_patient_concentration,young_patient_concentration
0,1003000597,OK,2022,0,1.692306,0.166923,0.166923,1.051990,1.692306,0.166923,...,3.003923,True,True,4.288024,True,False,-0.187419,False,0.0,0.0
1,1003000597,OK,2023,0,23.833223,-0.150274,-0.226470,1.001578,0.112054,-0.475409,...,1.564890,False,True,2.626744,True,False,-0.159271,False,0.0,0.0
2,1003000902,KY,2021,0,155.711788,-2.380244,-0.336014,1.000711,0.087436,-0.660212,...,-0.036876,False,False,-0.073170,False,False,-0.092180,False,0.0,0.0
3,1003000902,KY,2022,0,67.361088,-2.190131,-0.381036,1.000020,0.028794,-0.621005,...,-0.374303,False,False,0.108292,False,False,-0.098162,False,0.0,0.0
4,1003000902,KY,2023,0,40.802223,-1.466342,-0.355328,1.004687,0.098643,-0.463672,...,-0.049636,False,False,0.152416,False,False,-0.091872,False,0.0,0.0


In [7]:
df.columns.to_list()

['npi',
 'rfrg_prvdr_state_abrvtn',
 'year',
 'target',
 'avg_suplr_mdcr_pymt_amt_sum',
 'tot_suplr_mdcr_pymt_amt_hcpcs_rentl_ind_z_scr_sum',
 'tot_suplr_mdcr_pymt_amt_hcpcs_rentl_ind_z_scr_median',
 'avg_suplr_mdcr_alowd_amt_hcpcs_rentl_ind_rat_median',
 'avg_suplr_mdcr_pymt_amt_min',
 'tot_suplr_mdcr_pymt_amt_hcpcs_rentl_ind_z_scr_min',
 'avg_suplr_mdcr_alowd_amt_hcpcs_rentl_ind_rat_min',
 'avg_suplr_mdcr_pymt_amt_max',
 'tot_suplr_mdcr_pymt_amt_max',
 'tot_suplr_mdcr_pymt_amt_hcpcs_rentl_ind_z_scr_max',
 'avg_suplr_mdcr_alowd_amt_hcpcs_rentl_ind_rat_max',
 'tot_suplr_nonrntl_hcpcs_cds',
 'tot_suplr_rentl_hcpcs_cds',
 'tot_suplrs_sum',
 'tot_suplrs_median',
 'tot_suplr_benes_median',
 'tot_suplrs_min',
 'tot_suplr_clms_min',
 'tot_suplr_srvcs_min',
 'tot_suplrs_max',
 'tot_suplr_clms_max',
 'cnt_tot_suplr_srvcs_hcpcs_rentl_ind_pctl_abv_90',
 'accessories_for_oxygen_delivery_devices',
 'breathing_aids',
 'hospital_beds_and_associated_supplies',
 'humidifiers_and_nebulizers_with_relate

In [8]:
param_grid = {
    "model__C": [0.001],
    "model__penalty": ["l1", "l2", "elasticnet"],
    "model__solver": ["saga"],
    "model__l1_ratio": [0.0],
    "model__max_iter": [500],
    "model__class_weight": ["balanced", {1: 5000}],
}

In [9]:
###########################
# RUN HISTORY
############################

###### RUN 1
# Yes INNE, Cleanse, k_best_catboost
    #1884 seconds
# param_grid = {
#     "model__C": [0.001],
#     "model__penalty": ["l1", "l2", "elasticnet"],
#     "model__solver": ["saga"],
#     "model__l1_ratio": [0.0],
#     "model__max_iter": [500],
#     "model__class_weight": ["balanced", {1: 5000}],
# }
# Fitting 5 folds for each of 6 candidates, totalling 30 fits

In [10]:
# Optional: Add INNE anomaly features before training
# Hyperparameters default to the best values from standalone INNE tuning
# You can comment out this block if you want to test the model without INNE features
 
df = add_inne_features(
    df,

    # Default INNE parameters that were found to be best during standalone tuning.
    # You can adjust these if you want to test different INNE configurations.
    transaction_data_path="/dsa/groups/casestudycf25/team02/silver/cms_general_payments_anomaly_ready.csv",
    group_col="npi",
    year_col="year",
    txn_provider_col="covered_recipient_npi",
    txn_year_col="program_year",
    txn_target_col="target",
    txn_id_cols=["covered_recipient_npi", "record_id", "program_year"],
    n_estimators=100,
    max_samples=8,
    contamination=0.0005,
    top_k_features=15,
    null_fill=-1,
    verbose=1,
)

INNE: Loading /dsa/groups/casestudycf25/team02/silver/cms_general_payments_anomaly_ready.csv
INNE: 932908 transactions, 15 features selected
INNE: Fitted with n_estimators=100, max_samples=8, contamination=0.0005
INNE: Score range=[0.1916, 1.0000], P95 cutoff=0.9151
INNE: Rolled up to 158751 provider-years, 18285 with any above P95
INNE: Joined -> (140827, 219), 39586/140827 matched (28.1%)


In [11]:
pipe = MLPipeline(model_name="logistic_regression", target_col=target_col)

In [12]:
print(f"\n{'='*60}")
print(f"Running with feature_selection_method: k_best_catboost")
print(f"{'='*60}\n")

pipe.train(df, 
               param_grid=param_grid, 
               group_col="npi", 
               exclude_cols=exclude_cols, 
               undersample_method='CLEANSE', 
               feature_selection_method="k_best_catboost")


Running with feature_selection_method: k_best_catboost


Undersampling (CLEANSE) applied.
Class distribution after undersampling: {0: 65698, 1: 34}

Feature Selection (k_best_catboost) selected.
Fitting 5 folds for each of 6 candidates, totalling 30 fits

------------------------------------------------------------------------------------------------------------------------------------------------------


Model: logistic_regression
CV folds: 5  |  Scoring: average_precision
Best CV score: 0.0014 ± 0.0013 (std across folds)
Score range: min = 0.0005  max = 0.0041  cv = 0.9485 (std/mean)
Training time: 79.7 Seconds
Best params: {'model__C': 0.001, 'model__class_weight': {1: 5000}, 'model__l1_ratio': 0.0, 'model__max_iter': 500, 'model__penalty': 'l1', 'model__solver': 'saga'}

Per-fold breakdown:
  Fold          Score
  1            0.0041
  2            0.0008
  3            0.0005
  4            0.0008
  5            0.0009


-----------------------------------------------------------

In [17]:
print(f"\n{'='*60}")
print(f"Running with feature_selection_method: k_best_xgb")
print(f"{'='*60}\n")
    
pipe.train(df, 
               param_grid=param_grid, 
               group_col="npi", 
               exclude_cols=exclude_cols, 
               undersample_method='CLEANSE', 
               feature_selection_method="k_best_xgb")


Running with feature_selection_method: k_best_xgb

[CV 5/5] END model__C=0.001, model__class_weight=balanced, model__l1_ratio=0.0, model__max_iter=500, model__penalty=l1, model__solver=saga;, score=0.004 total time=   7.7s
[CV 4/5] END model__C=0.001, model__class_weight=balanced, model__l1_ratio=0.0, model__max_iter=500, model__penalty=l2, model__solver=saga;, score=0.001 total time=  16.7s
[CV 1/5] END model__C=0.001, model__class_weight={1: 5000}, model__l1_ratio=0.0, model__max_iter=500, model__penalty=l2, model__solver=saga;, score=0.003 total time=  13.8s
[CV 1/5] END model__C=0.001, model__class_weight=balanced, model__l1_ratio=0.0, model__max_iter=500, model__penalty=l1, model__solver=saga;, score=0.001 total time=   7.9s
[CV 5/5] END model__C=0.001, model__class_weight=balanced, model__l1_ratio=0.0, model__max_iter=500, model__penalty=l2, model__solver=saga;, score=0.002 total time=   4.6s
[CV 4/5] END model__C=0.001, model__class_weight=balanced, model__l1_ratio=0.0, model__

In [18]:
print(f"\n{'='*60}")
print(f"Running with feature_selection_method: k_best_lgbm")
print(f"{'='*60}\n")
    
pipe.train(df, 
               param_grid=param_grid, 
               group_col="npi", 
               exclude_cols=exclude_cols, 
               undersample_method='CLEANSE', 
               feature_selection_method="k_best_lgbm")


Running with feature_selection_method: k_best_lgbm

[CV 5/5] END model__C=0.001, model__class_weight=balanced, model__l1_ratio=0.0, model__max_iter=500, model__penalty=l1, model__solver=saga;, score=0.001 total time=   9.6s
[CV 5/5] END model__C=0.001, model__class_weight={1: 5000}, model__l1_ratio=0.0, model__max_iter=500, model__penalty=l1, model__solver=saga;, score=0.000 total time=  10.0s
[CV 3/5] END model__C=0.001, model__class_weight=balanced, model__l1_ratio=0.0, model__max_iter=500, model__penalty=l1, model__solver=saga;, score=0.000 total time=   3.9s
[CV 5/5] END model__C=0.001, model__class_weight=balanced, model__l1_ratio=0.0, model__max_iter=500, model__penalty=l2, model__solver=saga;, score=0.001 total time=   3.9s
[CV 5/5] END model__C=0.001, model__class_weight=balanced, model__l1_ratio=0.0, model__max_iter=500, model__penalty=elasticnet, model__solver=saga;, score=0.001 total time=   4.1s
[CV 2/5] END model__C=0.001, model__class_weight={1: 5000}, model__l1_ratio=0.

In [15]:
print(f"\n{'='*60}")
print(f"Running with feature_selection_method: k_best_decision_tree")
print(f"{'='*60}\n")
    
pipe.train(df, 
               param_grid=param_grid, 
               group_col="npi", 
               exclude_cols=exclude_cols, 
               undersample_method='CLEANSE', 
               feature_selection_method="k_best_decision_tree")

In [16]:
# for feature_selection_m in select_methods:
#     print(f"\n{'='*60}")
#     print(f"Running with feature_selection_method: {feature_selection_m}")
#     print(f"{'='*60}\n")
    
#     pipe.train(df, 
#                param_grid=param_grid, 
#                group_col="npi", 
#                exclude_cols=exclude_cols, 
#                undersample_method='CLEANSE', 
#                feature_selection_method=feature_selection_m)